# D126 — Exercise: Assertions and Unit Testing

## Order Fulfilment Challenge (60 minutes)

Test an existing order-fulfilment module using plain Python assertions and the built-in `unittest` framework.

> The application code is supplied. Your task is to design and write tests for it. This is **not** a test-driven development exercise.

## Learning goals

By completing this exercise, you will practise:

- writing clear plain `assert` statements
- using Arrange, Act, Assert
- testing normal, boundary, and invalid inputs
- covering `if`/`elif`/`else` branches
- checking exception types and messages
- organizing tests with `unittest.TestCase` and `setUp`
- building and running a test suite

Recommended timing: 10 minutes to inspect and plan, 15 minutes for Part A, 30 minutes for Part B, and 5 minutes to run and review.

## Scenario and business rules

An online store needs reliable calculations before it sends an order to the warehouse.

### `OrderItem`

- Quantity must be greater than zero; otherwise raise `ValueError("Quantity must be positive")`.
- Unit price cannot be negative; otherwise raise `ValueError("Unit price cannot be negative")`.
- `amount` is `quantity * unit_price`.

### `Order`

- A new order starts empty with status `"draft"`.
- Only an `OrderItem` can be added; any other value raises `TypeError("item must be an OrderItem")`.
- Adding an existing SKU combines its quantity. The original unit price stays unchanged.
- `subtotal` is the sum of all item amounts.
- Discount rate is 10% for subtotals from ₹2,000 through ₹4,999.99, and 15% from ₹5,000. Below ₹2,000 it is 0%.
- Shipping is ₹120 when the discounted total is below ₹3,000; otherwise it is free. An empty order has no shipping charge.
- Tax is 18% of the amount after discount. Shipping is not taxed.
- `grand_total = discounted total + tax + shipping`.
- Checkout of an empty order raises `ValueError("Cannot checkout an empty order")`.
- A successful checkout changes status to `"confirmed"` and returns a summary dictionary.
- A confirmed order cannot be changed; adding an item raises `RuntimeError("Confirmed order cannot be changed")`.

## Supplied application code

Run this cell before writing tests. Read the branches and validation carefully, but do not modify the implementation.

In [ ]:
class OrderItem:
    def __init__(self, sku, name, quantity, unit_price):
        if quantity <= 0:
            raise ValueError("Quantity must be positive")
        if unit_price < 0:
            raise ValueError("Unit price cannot be negative")
        self.sku = sku
        self.name = name
        self.quantity = quantity
        self.unit_price = unit_price

    @property
    def amount(self):
        return self.quantity * self.unit_price


class Order:
    def __init__(self, order_id):
        self.order_id = order_id
        self.items = {}
        self.status = "draft"

    def add_item(self, item):
        if self.status == "confirmed":
            raise RuntimeError("Confirmed order cannot be changed")
        if not isinstance(item, OrderItem):
            raise TypeError("item must be an OrderItem")
        if item.sku in self.items:
            self.items[item.sku].quantity += item.quantity
        else:
            self.items[item.sku] = item

    @property
    def items_count(self):
        return sum(item.quantity for item in self.items.values())

    @property
    def subtotal(self):
        return sum(item.amount for item in self.items.values())

    @property
    def discount_rate(self):
        if self.subtotal >= 5000:
            return 0.15
        if self.subtotal >= 2000:
            return 0.10
        return 0.0

    @property
    def discount_amount(self):
        return self.subtotal * self.discount_rate

    @property
    def discounted_total(self):
        return self.subtotal - self.discount_amount

    @property
    def shipping_fee(self):
        if not self.items:
            return 0
        return 0 if self.discounted_total >= 3000 else 120

    @property
    def tax_amount(self):
        return self.discounted_total * 0.18

    @property
    def grand_total(self):
        return self.discounted_total + self.tax_amount + self.shipping_fee

    def checkout(self):
        if not self.items:
            raise ValueError("Cannot checkout an empty order")
        self.status = "confirmed"
        return {
            "order_id": self.order_id,
            "status": self.status,
            "items_count": self.items_count,
            "total": self.grand_total,
        }

## Test plan — complete before coding (about 10 minutes)

List at least **12 test cases**. Include normal values, exact boundaries, every important branch, state changes, and invalid inputs.

| # | Unit/behavior | Input or setup | Expected result | Category |
|---:|---|---|---|---|
| 1 | Example: item amount | quantity 2, price 750 | 1500 | normal |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |
| 6 |  |  |  |  |
| 7 |  |  |  |  |
| 8 |  |  |  |  |
| 9 |  |  |  |  |
| 10 |  |  |  |  |
| 11 |  |  |  |  |
| 12 |  |  |  |  |

## Part A — Plain assertions (15 minutes)

Write assertions for all six tasks below. Use separate assertions when separate failures matter.

1. A new order is a `draft`, contains an empty dictionary, has zero items, and has total zero.
2. An item with quantity 2 and unit price 750 has amount 1500.
3. After adding a keyboard (2 × ₹750) and mouse (1 × ₹500), the order has 3 units, 2 unique SKUs, and subtotal ₹2,000.
4. Adding another keyboard with the same SKU combines quantity to 3 and keeps the original ₹750 unit price.
5. For the ₹2,000 subtotal order, check the discount rate, discount amount, shipping fee, tax, and grand total. Use a tolerance for calculated decimal values.
6. Checkout returns a dictionary containing the expected ID, status, item count, and total; the order status also becomes `confirmed`.

In [ ]:
# Part A starter — replace the TODO comments with your assertions.

# 1. New-order state
order = Order("ORD-101")
# TODO

# 2. Item amount
keyboard = OrderItem("KEY-1", "Keyboard", 2, 750)
# TODO

# 3. Multiple items and totals
order.add_item(keyboard)
order.add_item(OrderItem("MOU-1", "Mouse", 1, 500))
# TODO

# 4. Duplicate SKU
duplicate_order = Order("ORD-102")
duplicate_order.add_item(OrderItem("KEY-1", "Keyboard", 2, 750))
duplicate_order.add_item(OrderItem("KEY-1", "Keyboard", 1, 999))
# TODO

# 5. Pricing branches for the ₹2,000 order
# Hint: expected discounted total=1800, tax=324, grand total=2244.
# TODO

# 6. Checkout result and changed state
summary = order.checkout()
# TODO

print("Part A assertions passed")

## Part B — `unittest` test suite (30 minutes)

Complete the test class. Keep Arrange, Act, and Assert visible in each method.

Required coverage:

1. `OrderItem.amount` for a normal item.
2. Zero quantity raises `ValueError` with the exact message.
3. Negative unit price raises `ValueError`.
4. Adding a new item updates count and subtotal.
5. Adding the same SKU combines quantity and preserves the first price.
6. A non-`OrderItem` value raises `TypeError` with the exact message.
7. Discount boundaries: subtotal ₹1,999 has 0%, ₹2,000 has 10%, and ₹5,000 has 15%.
8. Shipping branches: empty order ₹0, discounted total below ₹3,000 costs ₹120, and discounted total exactly ₹3,000 is free.
9. Checkout of an empty order raises `ValueError` with the exact message.
10. Successful checkout returns the expected summary and changes state.
11. Adding an item after checkout raises `RuntimeError`.
12. Grand total is correct for subtotal ₹5,000: discount ₹750, tax ₹765, shipping ₹0, total ₹5,015.

> Boundary hint: create the exact shipping-threshold subtotal with `10000 / 3`. It receives a 10% discount and produces a discounted total of `3000`. Use `assertAlmostEqual` for floating-point comparisons.

In [ ]:
import unittest


class TestOrderFulfilment(unittest.TestCase):
    def setUp(self):
        self.order = Order("ORD-TEST")
        self.keyboard = OrderItem("KEY-1", "Keyboard", 2, 750)
        self.mouse = OrderItem("MOU-1", "Mouse", 1, 500)

    def test_item_amount_is_quantity_times_unit_price(self):
        # TODO
        pass

    def test_zero_quantity_raises_value_error(self):
        # TODO: also check the exact exception message
        pass

    def test_negative_unit_price_raises_value_error(self):
        # TODO
        pass

    def test_add_new_item_updates_count_and_subtotal(self):
        # TODO
        pass

    def test_duplicate_sku_combines_quantity_and_keeps_first_price(self):
        # TODO
        pass

    def test_add_rejects_non_order_item(self):
        # TODO: also check the exact exception message
        pass

    def test_discount_rate_at_all_boundaries(self):
        # TODO: use subTest or three separate assertions/orders
        pass

    def test_shipping_fee_for_empty_below_limit_and_at_limit(self):
        # TODO: use assertAlmostEqual for the exact discounted-total boundary
        pass

    def test_empty_order_cannot_checkout(self):
        # TODO: also check the exact exception message
        pass

    def test_checkout_returns_summary_and_confirms_order(self):
        # TODO
        pass

    def test_confirmed_order_cannot_be_changed(self):
        # TODO
        pass

    def test_grand_total_for_five_thousand_subtotal(self):
        # TODO: check subtotal, rate, discount, tax, shipping, and total
        pass

## Build and run the suite

Run this only after completing the test methods. `verbosity=2` shows the behavior described by every test name.

In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestOrderFulfilment)
result = unittest.TextTestRunner(verbosity=2).run(suite)

assert result.testsRun >= 12, "Complete all 12 required test methods"
assert result.wasSuccessful(), "Fix the failing tests before submission"
print("All required unit tests passed")

## Review questions (5 minutes)

Answer briefly:

1. Why do ₹1,999, ₹2,000, and ₹5,000 all matter for discount testing?
2. Why is the shipping boundary based on `discounted_total`, not `subtotal`?
3. Which tests verify returned values, and which verify changed object state?
4. Why should calculated decimal values use `assertAlmostEqual` instead of exact equality?
5. What would a test name such as `test_1` fail to communicate?

## Submission checklist

- [ ] Test plan contains at least 12 meaningful cases.
- [ ] Part A contains assertions for all six tasks.
- [ ] All 12 `unittest` methods are implemented; no `pass` remains.
- [ ] Normal, boundary, branch, invalid-input, and state-change behavior is covered.
- [ ] Expected exceptions include checks for important messages.
- [ ] Floating-point values use a tolerance or `assertAlmostEqual`.
- [ ] Every test has a descriptive name and can run independently.
- [ ] The final suite reports all tests passing.

## Optional stretch tasks

Attempt these only after the required suite passes:

1. Use `subTest` to test all discount boundaries from a table of inputs and expected rates.
2. Prove that two orders created in separate tests do not share item state.
3. Add a test confirming that free items (unit price zero) are valid.
4. Identify one business rule that is currently unspecified and write the question you would ask the product owner.

Do not change the supplied application code for the core exercise.